# Diagnostic Reasoning Assistant (DDXPlus) — Notebook 03: RAG & Explainability

**Grounded explanations for Top‑k diagnoses + next-question selection (TF‑IDF baseline), integrated with the saved token LR + IG policy**

**Author:** Julia Parnis   
**Date:** March 1, 2026  

## Project Overview

**Project goal:** Build an AI-powered diagnostic assistant that provides a *ranked differential* through iterative questioning, with a focus on improving detection of rare / overlooked conditions.

**Scope of this notebook:** Integrate a Retrieval-Augmented Generation (RAG) layer on top of the trained token-level Logistic Regression model to enable:
- Natural-language question phrasing (ask → patient answers → re-rank)
- Evidence-backed explainability (why this disease? what does the evidence mean?)
- Cited differential diagnosis summaries grounded in the DDXPlus knowledge base

This notebook focuses on *retrieval pipeline design, prompt engineering, and end-to-end session loop validation*.

**Planned (later phases):**
- Streamlit / API deployment (interactive web interface)
- Advanced question selection policies (reinforcement learning / active learning)
- Evaluation of RAG output quality (faithfulness, citation accuracy, clinical coherence)

**Inputs from Notebook 02:**
- `logreg_token_calibrated.joblib` — calibrated token model for probability ranking
- `preprocessors_token.joblib` — mlb_token, ohe, scaler, label_encoder
- `policy_artifacts.joblib` — p_e_given_d, evidence_bases, evidence_index
- `.py` modules in `assistant/` — types, encoding, inference, policy, artifacts

**Dataset:** DDXPlus (synthetic) — ~1.3M patient cases, 49 pathologies, evidence-based features  
**Source:** Hugging Face: `aai530-group6/ddxplus`  
**Primary reference:** Fansi Tchango et al. (2022)

> **Important:** DDXPlus is synthetic (generated from medical knowledge bases + an AD system).  
> This notebook is for research/education only; it does **not** produce a deployable clinical tool.

## Notebook 03 objective: RAG-powered explainability and iterative questioning

**Objective:** Build and validate a Retrieval-Augmented Generation layer on top of the trained token-level ML model to enable a natural-language diagnostic assistant, emphasising:
- Grounded question phrasing (questions generated from retrieved evidence descriptions, not hardcoded)
- Evidence-backed explainability (cited reasoning for the ranked differential)
- End-to-end session loop (ask → patient answers → re-rank → next question)
- Faithful retrieval (retrieved context is relevant, non-hallucinated, and traceable to source)

**Success criteria (for this notebook):**
- RAG pipeline runs end-to-end from a cold `DiagnosisSession` to a cited differential summary
- Questions are phrased in natural language and grounded in `release_evidences.json`
- Top-k differential is explained with at least one retrieved supporting fact per disease
- Session loop completes at least 5 question–answer turns without errors
- All prompts, retrievals, and outputs are logged and saved to `outputs/rag/`


# Notebook 03 — RAG Explanations for the DDXPlus Diagnostic Reasoning Assistant

## What we are building
We are adding a **local Retrieval-Augmented Generation (RAG)** layer that produces:
1) grounded explanations for **why the current Top‑k diagnoses** are ranked highly, and  
2) grounded explanations for **why the next question** was selected by the information‑gain policy.

## What stays fixed (no retraining)
- The ranking engine remains the **Logistic Regression token/value-level model** (972 token vocab; 978 total features with demographics + numeric complexity).
- The question policy remains **base-level (223 evidences)** using the likelihood table `p_e_given_d = P(e=1 | disease)`.
- For policy decisions, we prefer **calibrated probabilities** when available because information gain depends on meaningful uncertainty estimates.

## Data sources for retrieval (local, reproducible)
We build a retrieval corpus from:
- `release_evidences.json` (evidence IDs → question text, data type, default values, value meanings)
- `release_conditions.json` (condition names + symptom/antecedent evidence sets, severity, ICD hints)

Optional: we may add a small curated “mini-manual” layer (short markdown notes per condition) to improve explanation quality without requiring any external web access.

## Safety / scope
This project is **educational only** and uses the **synthetic DDXPlus dataset**. Explanations are phrased as learning support (how evidence patterns relate to diagnoses in this dataset and in general) and are not medical advice.

# Workplan (Notebook 03)

## 0) Setup and reproducibility
- Define paths (`data/`, `models/`, `outputs/rag/`) and create output folders.
- Load saved artifacts (preprocessors, calibrated token LR when available, policy table).
- Confirm feature shapes and that the pipeline runs without retraining.

## 1) Load dataset metadata (the retrieval “ground truth”)
- Load `release_evidences.json` (223 evidence bases).
- Load `release_conditions.json` (49 conditions).
- Build helper functions to format evidence definitions and condition summaries.

## 2) Build the local retrieval corpus (TF‑IDF baseline)
- Create documents for:
  - each evidence base (question text + type + default/value meanings)
  - each condition (name + key symptoms/antecedents listed in the metadata)
- Store document metadata (doc_type, evidence_id / condition_id, fields used).
- Fit a TF‑IDF vectorizer and persist the corpus + vectors to `outputs/rag/`.

## 3) (Optional) Add a curated mini-manual layer (49 conditions)
- Create one short markdown note per condition:
  - “what it is” (1–2 sentences)
  - “typical evidence themes” (based on dataset metadata)
  - “common confusions” (dataset-adjacent, educational framing)
- Include these notes as additional retrievable documents.

## 4) Retrieval API (clean interface used by the loop)
- Implement `retrieve(query, top_n)` returning:
  - ranked docs + similarity scores
  - doc metadata for grounding/provenance
- Add lightweight tests: deterministic results given fixed corpus.

## 5) Grounded explanation templates
- Template A: **“Why these Top‑k diagnoses now?”**
  - Use: posterior Top‑k + retrieved condition summaries + retrieved evidence definitions.
- Template B: **“Why ask this next question?”**
  - Use: chosen evidence base + question text + IG score + how it separates top diagnoses.

## 6) Integrate with the existing assistant loop
- Session state → token inference → posterior → IG policy → chosen question.
- Run retrieval to ground:
  - Top‑k explanation
  - next-question explanation
- Keep outputs consistent with the smoke test expectations (no retraining, same artifacts).

## 7) Demo trace + saved outputs
- Run a short simulated session (a few turns).
- Save:
  - retrieved contexts
  - explanations
  - a compact “trace log” for debugging / demo.

## 0) Setup & reproducibility

This section configures paths, seeds, display settings, and figure defaults (Zoom-friendly) consistent with Notebook 02.  
It also creates the new stage folders: `outputs/rag/` and `figures/rag/`.

In [ ]:
# ============================================================
# 0.1 Imports + project-root check + src import
# ============================================================

from __future__ import annotations

import os
import sys
import random
import json
from pathlib import Path
from dataclasses import dataclass, field
import warnings
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Visualization (match Notebook 02)
import matplotlib.pyplot as plt
from cycler import cycler

# Artifacts / ML utilities (needed later in this notebook)
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Silence noisy warnings (keep notebook readable)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
# -------------------------
# Project root detection
# -------------------------
def find_project_root(start: Path | None = None) -> Path:
    """
    Find repo root by walking upward until data/ and models/ are found.
    Works whether the notebook is run from repo root or from notebooks/.
    """
    start = Path.cwd() if start is None else start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "models").exists():
            return p
    # Fallback: if running in a minimal environment, treat cwd as root
    return start


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"

print(f"✓ PROJECT_ROOT = {PROJECT_ROOT}")

# Ensure src/ is importable when running from notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

✓ PROJECT_ROOT = c:\Julia\Ironhack\Week20-24_Final_project\diagnostic-reasoning-assistant


In [6]:
# Quick check-up
from src.artifacts import load_json  # noqa: E402
print("✓ src imports OK")

✓ src imports OK


In [5]:
# ============================================================
# 0.2 Reproducibility: seeds + version snapshot
# ============================================================

random_seed = 42

os.environ["PYTHONHASHSEED"] = str(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

print(f"✓ Random seed set to: {random_seed}")

# Capture a small environment snapshot (helps later when debugging)
import sklearn  # local import to keep import cell tidy

env_info = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

print("Environment snapshot:")
for k, v in env_info.items():
    print(f"  - {k}: {v}")

✓ Random seed set to: 42
Environment snapshot:
  - python: 3.12.7
  - numpy: 2.4.2
  - pandas: 3.0.0
  - scikit_learn: 1.8.0
  - joblib: 1.5.3


In [8]:
# ============================================================
# 0.3 Config (Notebook-02 style) + create rag folders
# ============================================================

@dataclass
class Config:
    """
    Central configuration for this project.

    Folder strategy:
    - Fixed root folders: outputs/, models/, figures/
    - Subfolders: outputs/{stage}/, figures/{stage}/
    - File naming: stable filenames (no date tagging)
    """

    # ── Data paths ───────────────────────────────────────────
    data_dir: Path            = Path("data")
    evidence_map_path: Path   = Path("data/release_evidences.json")
    conditions_map_path: Path = Path("data/release_conditions.json")

    # ── Output folders ───────────────────────────────────────
    output_dir: Path  = Path("outputs")
    models_dir: Path  = Path("models")
    figures_dir: Path = Path("figures")

    # ── Reproducibility ──────────────────────────────────────
    random_seed: int = random_seed


config = Config()

# Ensure stage folders exist (Notebook 02 created eda/ml; Notebook 03 adds rag)
for stage in ["eda", "ml", "rag"]:
    (config.output_dir / stage).mkdir(parents=True, exist_ok=True)
    (config.figures_dir / stage).mkdir(parents=True, exist_ok=True)

config.models_dir.mkdir(parents=True, exist_ok=True)

print("✓ Directories ready:")
print(f"  - {config.output_dir / 'rag'}")
print(f"  - {config.figures_dir / 'rag'}")
print(f"  - {config.models_dir}")

✓ Directories ready:
  - outputs\rag
  - figures\rag
  - models


In [9]:
# ============================================================
# 0.4 Visualization Settings (same as Notebook 02)
# ============================================================

# Apply clean seaborn style (no seaborn import needed)
plt.style.use("seaborn-v0_8-white")

# IBM Design colorblind-safe palette
IBM_COLORS = {
    "blue": "#648FFF",
    "purple": "#785EF0",
    "magenta": "#DC267F",
    "orange": "#FE6100",
    "yellow": "#FFB000",
    "teal": "#06A39B",
    "gray": "#5F6368",
}

# Figure defaults (presentation-optimized)
plt.rcParams.update({
    # Figure size and resolution
    "figure.figsize": (6, 4),          # Good for 2+ figs per slide
    "figure.dpi": 120,                 # Screen display
    "savefig.dpi": 300,                # High-quality save

    # Font sizes (readable on Zoom)
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,

    # Appearance (subtle, professional)
    "axes.edgecolor": IBM_COLORS["gray"],
    "axes.linewidth": 1.2,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.8,
    "grid.alpha": 0.6,
})

# Set IBM color cycle (for multi-line plots)
plt.rcParams["axes.prop_cycle"] = cycler(color=[
    IBM_COLORS["blue"],
    IBM_COLORS["teal"],
    IBM_COLORS["purple"],
    IBM_COLORS["magenta"],
    IBM_COLORS["orange"],
])

print("✓ Plot style configured (Notebook-02 consistent)")

# -------------------------
# Display settings (Notebook 02 style)
# -------------------------
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

print("✓ Display settings configured")

✓ Plot style configured (Notebook-02 consistent)
✓ Display settings configured


In [10]:
# ============================================================
# 0.5 Save helpers (same as Notebook 02)
# ============================================================

def save_fig(fig, filename: str, stage: str = "ml"):
    """
    Save figure into figures/{stage}/filename.

    stage: "eda" or "ml" or "rag"
    """
    out_dir = config.figures_dir / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"✓ Saved: {path}")


def save_table_csv(df: pd.DataFrame, filename: str, stage: str = "ml", index: bool = False):
    out_dir = config.output_dir / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    df.to_csv(path, index=index)
    print(f"✓ Saved: {path}")

## Step 1 — Local retrieval corpus (DDXPlus metadata)

We build a **local** corpus from:
- `release_evidences.json` (223 evidence definitions: question text, data type, default value)
- `release_conditions.json` (49 condition entries: ICD10, severity, symptom/antecedent evidence sets)

This baseline corpus enables TF‑IDF retrieval (fast, reproducible) and is the foundation for grounded explanations.

In [11]:
# ============================================================
# 1.1 Load sources (DDXPlus mapping files)
# ============================================================

evidences_map = load_json(config.evidence_map_path)
conditions_map = load_json(config.conditions_map_path)

print(f"✓ Loaded release_evidences.json: {len(evidences_map)} evidences")
print(f"✓ Loaded release_conditions.json: {len(conditions_map)} conditions")

# Basic sanity checks (expected: 223 evidences, 49 conditions)
assert len(evidences_map) == 223, "Unexpected number of evidences (expected 223)."
assert len(conditions_map) == 49, "Unexpected number of conditions (expected 49)."

✓ Loaded release_evidences.json: 223 evidences
✓ Loaded release_conditions.json: 49 conditions
